<a href="https://colab.research.google.com/github/ChenHY1217/Projects-In-MLAI/blob/main/ProjectsinMLAIhw5_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNNs, AEs, GANs, Attention Mechanism

## Task 3 - NLP and Attention Mechanism

### Part 1 - Scaled dot-product attention

In [ ]:
import numpy as np

def softmax(x, axis=-1):

  e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
  return e_x / e_x.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
  """
  Computes the scaled dot product attention.

  Args:
    Q: Query matrix of shape (..., seq_len, d_k).
    K: Key matrix of shape (..., seq_len, d_k).
    V: Value matrix of shape (..., seq_len, d_k).
    mask: Mask matrix of shape (..., seq_len, seq_len).

  Returns:
    Output matrix of shape (..., seq_len, d_k).
  """

  d_k = Q.shape[-1]

  scaled_scores = np.matmul(Q, K.swapaxes(-2, -1)) / np.sqrt(d_k)

  if mask is not None:
    scaled_scores += mask * -1e9

  attention_weights = softmax(scaled_scores)

  output = np.matmul(attention_weights, V)

  return output, attention_weights

In [ ]:
seq_len = 3
d_k = 4
np.random.seed(42)

# Dummy Q, K, V matrices
Q = np.random.rand(seq_len, d_k)
K = np.random.rand(seq_len, d_k)
V = np.random.rand(seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

print("Attention Weights:\n", weights)
print("\nSum of weights along each row (should be 1.0):\n", np.sum(weights, axis=-1))
print("\nOutput Context Vector:\n", output)

Attention Weights:
 [[0.31164829 0.37066075 0.31769096]
 [0.32266579 0.33486977 0.34246444]
 [0.33283257 0.33507568 0.33209175]]

Sum of weights along each row (should be 1.0):
 [1. 1. 1.]

Output Context Vector:
 [[0.38238456 0.56336845 0.59419359 0.48028741]
 [0.36781777 0.59386381 0.59857094 0.49987658]
 [0.37190176 0.5920136  0.59070988 0.49675455]]


### Part 2 - Encoder Decoder seq2seq

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ScaledDotProductAttention(nn.Module):
    def __init__(self, hidden_size):
        super(ScaledDotProductAttention, self).__init__()
        self.hidden_size = hidden_size

    def forward(self, query, keys, values, mask=None):
        # query shape: (batch_size, seq_len, hidden_size)
        # keys shape: (batch_size, seq_len, hidden_size)
        # values shape: (batch_size, seq_len, hidden_size)

        d_k = query.size(-1)

        # Calculate scores: Q * K^T
        scores = torch.bmm(query, keys.transpose(1, 2))

        # Scale the scores
        scaled_scores = scores / (d_k ** 0.5)

        # Apply mask
        if mask is not None:
            scaled_scores = scaled_scores.masked_fill(mask == 0, -1e9)

        # Softmax to get attention weights
        attn_weights = F.softmax(scaled_scores, dim=-1)

        # Multiply weights by Values to get the context vector
        output = torch.bmm(attn_weights, values)

        return output, attn_weights

Encoder

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        # Converts word indexes into dense vectors
        self.embedding = nn.Embedding(input_size, hidden_size)

        # The GRU layer
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.dropout(self.embedding(x))

        # outputs shape: (batch_size, seq_len, hidden_size)
        # hidden shape: (1, batch_size, hidden_size)
        outputs, hidden = self.gru(embedded)

        return outputs, hidden

Decoder

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.dropout = nn.Dropout(dropout_p)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

        # attention layer
        self.attention = ScaledDotProductAttention(hidden_size)

        # A linear layer to combine the GRU output and the attention context vector
        self.combine_layer = nn.Linear(hidden_size * 2, hidden_size)

        # Final classification layer to guess the translated word
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden, encoder_outputs, mask=None):
        # x (current word) shape: (batch_size, 1)
        # hidden (previous state) shape: (1, batch_size, hidden_size)
        # encoder_outputs shape: (batch_size, seq_len, hidden_size)

        embedded = self.dropout(self.embedding(x))

        gru_out, hidden = self.gru(embedded, hidden)

        # Calculate Attention
        # Query = gru_out, Keys = encoder_outputs, Values = encoder_outputs
        context, attn_weights = self.attention(gru_out, encoder_outputs, encoder_outputs, mask)

        # Concatenate the GRU output and the context vector
        concat_out = torch.cat((gru_out, context), dim=-1)

        # Pass through a linear layer and tanh activation
        combined_out = torch.tanh(self.combine_layer(concat_out))

        # Predict the next word
        prediction = F.log_softmax(self.out(combined_out), dim=-1)

        return prediction, hidden, attn_weights

Looking at the Luong attention encoder decoder architecture, I implemented the above seq2seq model. The encoder will run the inputs through a GRU and the decoder will pass its inputs through a GRU and combine that with the context vector and Attention weights calculated through our scaled dot product attention and then pass through a linear layer and activations. We rewrite our scaled dot product attention implementation as a pytorch class to integrate into the neural networks. Based on the Luong architecture, the Decoder's output is the Query and the Encoder's outputs are the keys and values.

### Part 3 - Training and Evaluating seq2seq

In [ ]:
import torch
import torch.nn as nn
import random

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src shape: (batch_size, src_len)
        # trg shape: (batch_size, trg_len)
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_size

        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        # Pass the source sentence through the Encoder
        encoder_outputs, hidden = self.encoder(src)

        # First input to the decoder is the <sos> (start of sentence) token
        # Assuming the first token in the target sequence is <sos>
        decoder_input = trg[:, 0].unsqueeze(1) # shape: (batch_size, 1)

        for t in range(1, trg_len):
            # Pass through Decoder
            output, hidden, _ = self.decoder(decoder_input, hidden, encoder_outputs)

            outputs[:, t, :] = output.squeeze(1)

            # Decide whether to use teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(2)

            # If teacher forcing, use actual next word as next input. Else, use the model's prediction
            decoder_input = trg[:, t].unsqueeze(1) if teacher_force else top1

        return outputs

In [ ]:
!pip install datasets

We will be using the Multi30k dataset and training the model for translation from English to German.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from collections import Counter

def tokenize_de(text):
    return text.lower().split()

def tokenize_en(text):
    return text.lower().split()

# Download the Multi30k dataset via Hugging Face
print("Downloading dataset...")
dataset = load_dataset("bentrevett/multi30k")
train_data_hf = dataset['train'].select(range(10000))
test_data_hf = dataset['test']

# Build vocabulary dictionary
print("Building vocabularies...")
def build_vocab(text_iterable, min_freq=2):
    counter = Counter()
    for tokens in text_iterable:
        counter.update(tokens)

    # Define our standard special tokens
    itos = ['<unk>', '<pad>', '<sos>', '<eos>']
    # Add words that appear at least 'min_freq' times
    itos.extend([token for token, freq in counter.items() if freq >= min_freq])

    # Create the reverse mapping (string -> integer)
    stoi = {token: i for i, token in enumerate(itos)}
    return stoi, itos

# Tokenize all training text to build the vocabulary
de_train_tokens = [tokenize_de(example['de']) for example in train_data_hf]
en_train_tokens = [tokenize_en(example['en']) for example in train_data_hf]

de_stoi, de_itos = build_vocab(de_train_tokens)
en_stoi, en_itos = build_vocab(en_train_tokens)

# Map our special tokens to their integer IDs
PAD_IDX = de_stoi['<pad>']
BOS_IDX = de_stoi['<sos>']
EOS_IDX = de_stoi['<eos>']
UNK_IDX = de_stoi['<unk>']

# Process into tensors
def process_data(hf_dataset, de_tokens_list, en_tokens_list):
    data = []
    for i in range(len(hf_dataset)):
        # Convert German text to tensor, adding <sos> at the start and <eos> at the end
        de_tensor = torch.tensor([BOS_IDX] + [de_stoi.get(token, UNK_IDX) for token in de_tokens_list[i]] + [EOS_IDX], dtype=torch.long)
        # Convert English text to tensor
        en_tensor = torch.tensor([BOS_IDX] + [en_stoi.get(token, UNK_IDX) for token in en_tokens_list[i]] + [EOS_IDX], dtype=torch.long)

        data.append((de_tensor, en_tensor))
    return data

print("Processing training data...")
train_data = process_data(train_data_hf, de_train_tokens, en_train_tokens)

print("Processing test data...")
test_de_tokens = [tokenize_de(example['de']) for example in test_data_hf]
test_en_tokens = [tokenize_en(example['en']) for example in test_data_hf]
test_data = process_data(test_data_hf, test_de_tokens, test_en_tokens)

# Batch the data
def generate_batch(data_batch):
    de_batch, en_batch = [], []
    for de_item, en_item in data_batch:
        de_batch.append(de_item)
        en_batch.append(en_item)

    # Pad the sequences so every sentence in a batch is the same length
    de_batch = pad_sequence(de_batch, padding_value=PAD_IDX, batch_first=True)
    en_batch = pad_sequence(en_batch, padding_value=PAD_IDX, batch_first=True)
    return de_batch, en_batch

BATCH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=generate_batch)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=generate_batch)

print(f"Success! Loaded {len(train_loader.dataset)} training pairs.")

Building vocabularies...
Processing training data...
Processing test data...
Success! Loaded 10000 training pairs.


In [ ]:
import torch.optim as optim
import torch.nn as nn
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize the models using the length of our new dictionaries
INPUT_DIM = len(en_stoi)
OUTPUT_DIM = len(de_stoi)
HIDDEN_DIM = 256

enc = EncoderRNN(INPUT_DIM, HIDDEN_DIM).to(device)
dec = DecoderRNN(HIDDEN_DIM, OUTPUT_DIM).to(device)
model = Seq2Seq(enc, dec, device).to(device)

# Setup Optimizer and Loss
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for de_batch, en_batch in train_loader:

        # English is our Source (src), German is our Target (trg)
        src = en_batch.to(device)
        trg = de_batch.to(device)

        optimizer.zero_grad()

        output = model(src, trg)

        # Flatten the outputs for CrossEntropyLoss
        output_dim = output.shape[-1]
        output = output[:, 1:, :].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()

    print(f'Epoch: {epoch+1:02} | Train Loss: {epoch_loss / len(train_loader):.3f}')

Epoch: 01 | Train Loss: 5.577
Epoch: 02 | Train Loss: 4.916
Epoch: 03 | Train Loss: 4.493
Epoch: 04 | Train Loss: 4.093
Epoch: 05 | Train Loss: 3.742
Epoch: 06 | Train Loss: 3.431
Epoch: 07 | Train Loss: 3.186
Epoch: 08 | Train Loss: 3.007
Epoch: 09 | Train Loss: 2.796
Epoch: 10 | Train Loss: 2.631


In [ ]:
import nltk
from nltk.translate.bleu_score import corpus_bleu

def translate_sentence(model, sentence, en_stoi, de_itos, device, max_len=50):
    model.eval()

    # Tokenize the raw string
    tokens = tokenize_en(sentence)
    tokens = ['<sos>'] + tokens + ['<eos>']

    # Convert to integer indexes
    src_indexes = [en_stoi.get(token, UNK_IDX) for token in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src_tensor)

    trg_indexes = [BOS_IDX]

    # Generate the translation token by token
    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).unsqueeze(0).to(device)

        with torch.no_grad():
            output, hidden, _ = model.decoder(trg_tensor, hidden, encoder_outputs)

        pred_token = output.argmax(2).item()
        trg_indexes.append(pred_token)

        if pred_token == EOS_IDX:
            break

    # Convert integer indexes back to German words
    trg_tokens = [de_itos[i] for i in trg_indexes]
    return trg_tokens[1:-1] # Return the sentence without <sos> and <eos>

def calculate_bleu(hf_dataset, model, en_stoi, de_itos, device):
    trgs = []
    pred_trgs = []

    # Iterate over the Hugging Face test dataset
    for datum in hf_dataset:
        src = datum['en']
        trg = datum['de']

        # Get the model's prediction
        pred_trg = translate_sentence(model, src, en_stoi, de_itos, device)
        pred_trgs.append(pred_trg)

        # Tokenize the actual target sentence for comparison
        trg_tokens = tokenize_de(trg)
        trgs.append([trg_tokens])

    # Calculate BLEU score using NLTK
    return corpus_bleu(trgs, pred_trgs)

# Run the evaluation
print("Calculating BLEU Score...")
bleu = calculate_bleu(test_data_hf, model, en_stoi, de_itos, device)
print(f'Final Test BLEU score = {bleu*100:.2f}')

Calculating BLEU Score...
Final Test BLEU score = 16.55


### Part 4 - Simplified Transformer Model

Positional Encoding

In [ ]:
import torch
import torch.nn as nn
import math

# --- Hyperparameters ---
EMBED_DIM = 64
NUM_HEADS = 2
FF_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.1

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        # Create a matrix of shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Apply sine to even indices and cosine to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Shape: (1, max_len, d_model)

        # save with model but don't update
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Add positional encodings to the token embeddings
        x = x + self.pe[:, :x.size(1), :]
        return x

Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        # Linear projections and reshape for multiple heads
        # Shape becomes: (batch_size, num_heads, seq_len, d_k)
        Q = self.q_linear(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.k_linear(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.v_linear(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            # We add a dimension for the heads: (batch_size, 1, seq_len, seq_len)
            mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)

        # Concatenate heads and pass through final linear layer
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.out_linear(context)

        return output

Feed Forward, Encoder, and Decoder Layers

In [ ]:
class PositionwiseFeedforward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionwiseFeedforward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedforward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Self-attention + Residual + Norm
        attn_out = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        # Feedforward + Residual + Norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedforward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, trg_mask):
        # Masked self-attention (autoregressive)
        attn_out = self.self_attn(x, x, x, trg_mask)
        x = self.norm1(x + self.dropout(attn_out))
        # Cross-attention over encoder outputs
        cross_out = self.cross_attn(q=x, k=enc_out, v=enc_out, mask=src_mask)
        x = self.norm2(x + self.dropout(cross_out))
        # Feedforward
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))
        return x

Full Transformer Architecture

In [ ]:
class SimplifiedTransformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model, num_heads, num_layers, d_ff, dropout=0.1):
        super(SimplifiedTransformer, self).__init__()

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.pe = PositionalEncoding(d_model)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc_out = nn.Linear(d_model, trg_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_masks(self, src, trg, pad_idx):
        # Source mask: hide padding tokens.
        src_mask = (src != pad_idx).unsqueeze(1)

        if trg is not None:
            # Target mask: hide padding and future tokens (causal mask)
            trg_pad_mask = (trg != pad_idx).unsqueeze(1)
            trg_len = trg.size(1)
            # Create a triangular matrix to mask future positions
            trg_sub_mask = torch.tril(torch.ones((trg_len, trg_len), device=src.device)).bool()
            trg_mask = trg_pad_mask & trg_sub_mask
            return src_mask, trg_mask

        return src_mask, None

    def forward(self, src, trg, pad_idx):
        src_mask, trg_mask = self.generate_masks(src, trg, pad_idx)

        # Encoder Pipeline
        enc_src = self.dropout(self.pe(self.src_embedding(src)))
        for layer in self.encoder_layers:
            enc_src = layer(enc_src, src_mask)

        # Decoder Pipeline
        dec_trg = self.dropout(self.pe(self.trg_embedding(trg)))
        for layer in self.decoder_layers:
            dec_trg = layer(dec_trg, enc_src, src_mask, trg_mask)

        # Final Output
        output = self.fc_out(dec_trg)
        return output

Since we are using CrossEntropyLoss, we will not include the SoftMax activation in the forward function but rather use log_softmax in inference to satisfy this requirement.

Training and Testing Transformer

In [ ]:
import torch.optim as optim
import torch.nn as nn
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize the models
INPUT_DIM = len(en_stoi)
OUTPUT_DIM = len(de_stoi)
EMBED_DIM = 64
NUM_HEADS = 2
NUM_LAYERS = 2
FF_DIM = 128

model = SimplifiedTransformer(
    src_vocab_size=INPUT_DIM,
    trg_vocab_size=OUTPUT_DIM,
    d_model=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=FF_DIM
).to(device)

# Setup Optimizer and Loss
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.0005)

epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for de_batch, en_batch in train_loader:
        # English is Source, German is Target
        src = en_batch.to(device)
        trg = de_batch.to(device)

        optimizer.zero_grad()

        # We feed the model everything EXCEPT the last token (<eos>)
        trg_input = trg[:, :-1]
        trg_expected = trg[:, 1:]

        # Forward pass
        output = model(src, trg_input, PAD_IDX)

        # Flatten outputs and expected targets for CrossEntropyLoss
        output_dim = output.shape[-1]
        output = output.reshape(-1, output_dim)
        trg_expected = trg_expected.reshape(-1)

        loss = criterion(output, trg_expected)
        loss.backward()

        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()

    print(f'Epoch: {epoch+1:02} | Train Loss: {epoch_loss / len(train_loader):.3f}')

Epoch: 01 | Train Loss: 6.739
Epoch: 02 | Train Loss: 5.247
Epoch: 03 | Train Loss: 4.773
Epoch: 04 | Train Loss: 4.441
Epoch: 05 | Train Loss: 4.191
Epoch: 06 | Train Loss: 3.994
Epoch: 07 | Train Loss: 3.838
Epoch: 08 | Train Loss: 3.695
Epoch: 09 | Train Loss: 3.580
Epoch: 10 | Train Loss: 3.466


In [ ]:
def translate_sentence(model, sentence, en_stoi, de_itos, device, max_len=50):
    model.eval()

    # Process the source sentence
    tokens = tokenize_en(sentence)
    tokens = ['<sos>'] + tokens + ['<eos>']
    src_indexes = [en_stoi.get(token, UNK_IDX) for token in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(0).to(device)

    # Start the target sequence with the <sos> token
    trg_indexes = [BOS_IDX]

    for i in range(max_len):
        trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(0).to(device)

        with torch.no_grad():
            # Pass both the source and the growing target sequence to the model
            output = model(src_tensor, trg_tensor, PAD_IDX)

        # Get the prediction for the newest token
        pred_token = output.argmax(2)[:, -1].item()
        trg_indexes.append(pred_token)

        # Stop if the model generates the <eos> token
        if pred_token == EOS_IDX:
            break

    # Convert indexes to words
    trg_tokens = [de_itos[i] for i in trg_indexes]
    return trg_tokens[1:-1]

def calculate_bleu_transformer(hf_dataset, model, en_stoi, de_itos, device):
    trgs = []
    pred_trgs = []

    # Iterate over the Hugging Face test dataset
    for datum in hf_dataset:
        src = datum['en']
        trg = datum['de']

        # Get the model's prediction
        pred_trg = translate_sentence(model, src, en_stoi, de_itos, device)
        pred_trgs.append(pred_trg)

        # Tokenize the actual target sentence for comparison
        trg_tokens = tokenize_de(trg)
        trgs.append([trg_tokens])

    # Calculate BLEU score using NLTK
    return corpus_bleu(trgs, pred_trgs)

# Run the evaluation
print("Calculating BLEU Score for Simplified Transformer...")
bleu = calculate_bleu_transformer(test_data_hf, model, en_stoi, de_itos, device)
print(f'Final Test BLEU score = {bleu*100:.2f}')

Calculating BLEU Score for Simplified Transformer...
Final Test BLEU score = 9.93


There are clear differences in the performance of the Simplified Transformer versus the Seq2Seq Model. We notice that the transformer model actually performed worst than the seq2seq model. This is likely because transformers are highly data-hungry. With the restriction to a small dataset, the Transformer does not have enough examples to learn. Meanwhile, the RNNs in the seq2seq model can learn even with small datasets. In addition, because we are using basic word-level tokenization, we are hurting the vocabulary. This decreases BLEU scores. These restrictions as well as the other simplifications all reduces the power of the transformer resulting in its inferior performance compared to RNNs.

The Transformer model trains significantly faster per epoch than se2seq model because of the lack of needing to wait sequentially.